# MRI/fMRI Spatial Foundations: NIfTI, Orientation and Coregistration

Учебный notebook проекта **NeuroAI Research Track** на данных  
**Wakeman & Henson multimodal neuroimaging dataset — OpenNeuro `ds000117`**.

Здесь мы последовательно разбираем:

- загрузку реальных BIDS/NIfTI данных;
- `voxel`, `volume`, `TR`, voxel size и field of view;
- affine и преобразование `voxel ↔ world coordinates`;
- orientation (`LAS`, `RAS`) и canonical orientation;
- различия T1w MRI и BOLD fMRI / EPI;
- mean EPI;
- rigid coregistration `mean EPI → T1w`;
- mutual information;
- resampling;
- visual quality control (QC).

> **Важно:** это учебный spatial pipeline, а не готовый preprocessing workflow для научного вывода.  
> Мы намеренно не выполняем здесь motion correction, susceptibility distortion correction, brain masking, normalization и другие шаги полноценного fMRI preprocessing.

## 1. Environment setup

Notebook рассчитан в первую очередь на Google Colab.  
Устанавливаем только библиотеки, которые реально используем в этом notebook.

In [ ]:
# Устанавливаем зависимости в текущее окружение notebook.
# %pip предпочтительнее !pip, потому что работает с Python environment текущего kernel.
%pip -q install nibabel boto3 dipy

In [ ]:
# Базовые импорты.
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np

from botocore import UNSIGNED
from botocore.config import Config
from nibabel.affines import apply_affine
from nibabel.orientations import aff2axcodes

In [ ]:
# В Colab используем /content; локально сохраняем временные данные внутри ../data/tmp.
DATA_DIR = (
    Path("/content/neuro_ai_data")
    if Path("/content").exists()
    else Path("../data/tmp")
)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Temporary data directory:", DATA_DIR)

## 2. OpenNeuro / BIDS: находим функциональные файлы

OpenNeuro публикует этот dataset через публичный S3 bucket.  
Подключаемся анонимно и смотрим содержимое `sub-01/ses-mri/func`.

Это одновременно показывает BIDS-структуру реального исследования:
`subject → session → modality → files`.

In [ ]:
# Создаём анонимный S3 client: credentials для публичного OpenNeuro bucket не нужны.
s3 = boto3.client(
    "s3",
    config=Config(signature_version=UNSIGNED),
)

bucket = "openneuro.org"
func_prefix = "ds000117/sub-01/ses-mri/func/"

# Получаем список файлов функциональной MRI-сессии sub-01.
response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix=func_prefix,
)

func_keys = [obj["Key"] for obj in response.get("Contents", [])]

# Для компактности показываем только run-01.
for key in func_keys:
    if "run-01" in key:
        print(key)

Для одного run нас особенно интересуют три BIDS-файла:

- `*_bold.nii.gz` — 4D BOLD data;
- `*_bold.json` — acquisition metadata;
- `*_events.tsv` — временная структура эксперимента.

В этом notebook работаем с `run-01_bold.nii.gz`.  
`events.tsv` будет отдельным следующим учебным блоком.

In [ ]:
# Явно задаём путь к run-01 BOLD, чтобы notebook не зависел от порядка файлов в S3 listing.
bold_key = (
    "ds000117/sub-01/ses-mri/func/"
    "sub-01_ses-mri_task-facerecognition_run-01_bold.nii.gz"
)

bold_path = DATA_DIR / "sub-01_run-01_bold.nii.gz"

# Скачиваем только один BOLD run, а не весь dataset.
if not bold_path.exists():
    s3.download_file(bucket, bold_key, str(bold_path))

print("BOLD file:", bold_path)

## 3. Загружаем BOLD NIfTI

`nib.load()` возвращает NIfTI image object.  
На этом этапе данные ещё не обязаны полностью загружаться в RAM: NiBabel может работать через proxy.

In [ ]:
# Загружаем 4D BOLD NIfTI.
func_img = nib.load(bold_path)

print("fMRI shape:", func_img.shape)
print("Voxel sizes + TR:", func_img.header.get_zooms())
print("Orientation:", aff2axcodes(func_img.affine))

Для `run-01` ожидаем:

- shape: `(64, 64, 33, 208)`;
- spatial voxel size: примерно `3 × 3 × 3.75 mm`;
- `TR = 2 s`.

То есть один volume — это `64 × 64 × 33`, а весь fMRI run — последовательность из 208 таких volumes.

In [ ]:
# Берём только первый 3D volume через data proxy.
# Так мы не загружаем весь 4D run в память только ради первой визуализации.
first_volume = np.asarray(func_img.dataobj[..., 0], dtype=np.float32)

print("First volume shape:", first_volume.shape)

In [ ]:
def plot_orthogonal_slices(volume, title_prefix):
    """Показывает центральные sagittal, coronal и axial срезы 3D volume."""
    x, y, z = (np.array(volume.shape) // 2).astype(int)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    # Axis 0 соответствует left-right направлению после учёта ориентации изображения.
    axes[0].imshow(volume[x, :, :].T, cmap="gray", origin="lower")
    axes[0].set_title(f"{title_prefix} — Sagittal")

    axes[1].imshow(volume[:, y, :].T, cmap="gray", origin="lower")
    axes[1].set_title(f"{title_prefix} — Coronal")

    axes[2].imshow(volume[:, :, z].T, cmap="gray", origin="lower")
    axes[2].set_title(f"{title_prefix} — Axial")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


plot_orthogonal_slices(first_volume, "BOLD volume 0")

## 4. Structural T1w MRI

Теперь загружаем структурное T1w изображение **того же участника**.

T1w даёт гораздо более детальную анатомию, чем EPI/BOLD, поэтому позже будет нашим
`static / fixed image` для coregistration.

In [ ]:
# Известный BIDS-путь к T1w sub-01.
t1w_key = "ds000117/sub-01/ses-mri/anat/sub-01_ses-mri_T1w.nii.gz"
t1w_path = DATA_DIR / "sub-01_T1w.nii.gz"

if not t1w_path.exists():
    s3.download_file(bucket, t1w_key, str(t1w_path))

anat_img = nib.load(t1w_path)

print("T1w shape:", anat_img.shape)
print("T1w voxel sizes:", anat_img.header.get_zooms()[:3])
print("T1w orientation:", aff2axcodes(anat_img.affine))

In [ ]:
# Загружаем T1w как float32: этого достаточно для нашей визуализации и registration,
# а памяти требуется меньше, чем для float64.
anat_data = anat_img.get_fdata(dtype=np.float32)

plot_orthogonal_slices(anat_data, "T1w")

### T1w vs BOLD

Здесь важно не путать **размер массива** и **физическое разрешение**.

- T1w: примерно `1 × 1 × 1 mm` voxels;
- fMRI/EPI: примерно `3 × 3 × 3.75 mm` voxels.

Более мелкая voxel grid T1w содержит больше пространственной детализации.  
Resampling fMRI на T1w grid позднее **не создаст** эту потерянную детализацию.

## 5. Affine: voxel coordinates ↔ world coordinates

Voxel index — это адрес внутри NumPy-массива.  
Affine связывает этот индекс с физическими координатами изображения, обычно в миллиметрах.

Проверим одну точку `fMRI voxel = [30, 30, 15]`.

In [ ]:
print("fMRI affine:")
print(func_img.affine)

print("\nT1w affine:")
print(anat_img.affine)

In [ ]:
# Выбираем один voxel в исходной fMRI grid.
func_voxel = np.array([30, 30, 15])

# Переводим voxel index fMRI в физические world coordinates.
world_coord = apply_affine(func_img.affine, func_voxel)

# Через inverse affine T1w находим, где та же world coordinate лежит в T1w grid.
anat_voxel = apply_affine(
    np.linalg.inv(anat_img.affine),
    world_coord,
)

print("fMRI voxel:", func_voxel)
print("World coordinates, mm:", world_coord)
print("T1w voxel coordinate:", anat_voxel)
print("Nearest T1w voxel:", np.round(anat_voxel).astype(int))

Ключевое ограничение:

> Одинаковая **header-derived world coordinate** в двух изображениях ещё не доказывает,
> что их анатомическое содержимое идеально совмещено.

Affine описывает геометрию файлов.  
**Registration** оценивает дополнительное преобразование по самому изображению.

## 6. Orientation и canonical RAS

Проверяем направления осей, зашитые в affine.

In [ ]:
print("Original fMRI orientation:", aff2axcodes(func_img.affine))
print("Original T1w orientation:", aff2axcodes(anat_img.affine))

В этих данных исходные orientations различаются:

- fMRI: `LAS`;
- T1w: `RAS`.

Это объясняет, почему одинаковые array indices могут визуально находиться с разных сторон.

Canonical orientation — это **переупорядочивание / flipping voxel axes**, а не registration.

In [ ]:
# Приводим оба изображения к ближайшей canonical (RAS+) ориентации.
# Это удобно для обучения и визуального сравнения, но НЕ является обязательным шагом
# registration: корректный affine уже несёт информацию об ориентации.
func_can = nib.as_closest_canonical(func_img)
anat_can = nib.as_closest_canonical(anat_img)

print("Canonical fMRI:", aff2axcodes(func_can.affine))
print("Canonical T1w:", aff2axcodes(anat_can.affine))

In [ ]:
# World coordinate не меняется от reorientation.
# Меняется только voxel index, которым эта физическая точка адресуется в новой grid.
func_voxel_can = apply_affine(
    np.linalg.inv(func_can.affine),
    world_coord,
)
anat_voxel_can = apply_affine(
    np.linalg.inv(anat_can.affine),
    world_coord,
)

print("Canonical fMRI voxel:", func_voxel_can)
print("Canonical T1w voxel:", anat_voxel_can)
print("Same world coordinate:", world_coord)

Итого:

`voxel index → affine → world coordinate`

а orientation говорит, в каких анатомических направлениях идут оси массива.

`canonical orientation ≠ registration`

## 7. Field of View (FOV)

Сравним approximate physical coverage вдоль voxel axes.

Здесь используем:

`number of voxels × voxel spacing`

Это полезная оценка acquisition coverage, но не мера «масштаба мозга».

In [ ]:
func_voxel_sizes = np.array(func_can.header.get_zooms()[:3], dtype=float)
anat_voxel_sizes = np.array(anat_can.header.get_zooms()[:3], dtype=float)

func_fov = np.array(func_can.shape[:3]) * func_voxel_sizes
anat_fov = np.array(anat_can.shape[:3]) * anat_voxel_sizes

print("fMRI voxel sizes, mm:", func_voxel_sizes)
print("T1w voxel sizes, mm:", anat_voxel_sizes)
print("Approx. fMRI FOV, mm:", func_fov)
print("Approx. T1w FOV, mm:", anat_fov)

Для этого run fMRI покрывает меньший физический объём, особенно по одной из spatial axes.

Это **limited FOV / coverage**, а не доказательство того, что мозг на fMRI «меньше».

Дополнительно EPI чувствительна к susceptibility artifacts около границ
`brain ↔ bone ↔ air`, особенно в orbitofrontal области. Возможны:

- geometric distortion;
- signal dropout.

Rigid registration не способна восстановить потерянный signal.

## 8. Mean EPI для coregistration

Для регистрации 4D run к T1w нужен один репрезентативный 3D functional image.

В учебном примере берём mean EPI: усредняем каждый voxel по времени.

> Ограничение: в реальном preprocessing обычно сначала учитывают head motion.
> Здесь mean EPI строится без motion correction только для понимания registration.

In [ ]:
# Загружаем canonical 4D fMRI как float32 и усредняем по temporal axis.
func_can_data = func_can.get_fdata(dtype=np.float32)
mean_epi = func_can_data.mean(axis=3)

# Canonical T1w также используем как float32.
anat_can_data = anat_can.get_fdata(dtype=np.float32)

# Registration terminology:
# static/fixed = reference image, которое оставляем в его пространстве;
# moving       = image, для которого ищем transformation.
static = anat_can_data
moving = mean_epi

static_affine = anat_can.affine
moving_affine = func_can.affine

print("Mean EPI / moving shape:", moving.shape)
print("T1w / static shape:", static.shape)

## 9. Initial alignment: centers of mass

Первый шаг — грубая инициализация.

Center-of-mass alignment выполняет только translation и даёт optimizеру стартовую точку.
Это ещё не полноценная rigid coregistration.

In [ ]:
from dipy.align.imaffine import transform_centers_of_mass

# Грубое начальное совмещение moving EPI с static T1w.
c_of_mass = transform_centers_of_mass(
    static,
    static_affine,
    moving,
    moving_affine,
)

# transform() применяет найденное преобразование и resamples moving image
# на voxel grid static image.
moving_com = c_of_mass.transform(moving)

print("Moving before:", moving.shape)
print("Static:", static.shape)
print("Moving after COM:", moving_com.shape)

In [ ]:
def normalize_for_display(data, low_percentile=2, high_percentile=98):
    """Robust normalization только для визуализации; исходные данные не изменяются."""
    low, high = np.percentile(data, [low_percentile, high_percentile])

    if high <= low:
        raise ValueError("Image has insufficient intensity range for display normalization.")

    clipped = np.clip(data, low, high)
    return (clipped - low) / (high - low)


static_norm = normalize_for_display(static)
moving_com_norm = normalize_for_display(moving_com)

In [ ]:
def plot_axial_overlay(static_image, moving_image, title, alpha=0.35):
    """Показывает central axial slice static, moving и их overlay."""
    z = static_image.shape[2] // 2

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(static_image[:, :, z].T, cmap="gray", origin="lower")
    axes[0].set_title("T1w — static")

    axes[1].imshow(moving_image[:, :, z].T, cmap="gray", origin="lower")
    axes[1].set_title("Mean EPI — moving")

    axes[2].imshow(static_image[:, :, z].T, cmap="gray", origin="lower")
    axes[2].imshow(
        moving_image[:, :, z].T,
        cmap="magma",
        origin="lower",
        alpha=alpha,
    )
    axes[2].set_title(title)

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


plot_axial_overlay(static_norm, moving_com_norm, "Overlay after COM")

После `transform()` output имеет T1w shape, но это **resampling**, а не увеличение реального
разрешения fMRI. Интерполяция не создаёт новых измеренных anatomical details.

## 10. Similarity metric: Mutual Information

T1w и EPI имеют разный intensity contrast, поэтому voxel-wise equality или обычная
линейная correlation — плохой критерий multimodal alignment.

**Mutual information (MI)** оценивает статистическую зависимость между intensity
distributions двух изображений.

При более согласованном anatomical alignment обычно возрастает MI.

`nbins=32` — это наша дискретизация интенсивностей для оценки joint/marginal
distributions, а не параметр MRI acquisition.

In [ ]:
from dipy.align.imaffine import AffineRegistration, MutualInformationMetric

# Используем все voxels для оценки mutual information.
metric = MutualInformationMetric(
    nbins=32,
    sampling_proportion=None,
)

# Multi-resolution optimization:
# coarse -> medium -> fine.
# Число iterations специально уменьшено относительно тяжёлого reference example,
# чтобы учебный notebook быстрее выполнялся в Colab.
level_iters = [1000, 100, 10]
sigmas = [3.0, 1.0, 0.0]
factors = [4, 2, 1]

affreg = AffineRegistration(
    metric=metric,
    level_iters=level_iters,
    sigmas=sigmas,
    factors=factors,
)

## 11. Translation optimization — 3 degrees of freedom

Сначала разрешаем optimizеру уточнить только `x/y/z` translation.

Стартуем не с identity, а с уже найденного center-of-mass transform.

In [ ]:
from dipy.align.transforms import TranslationTransform3D

translation = affreg.optimize(
    static,
    moving,
    TranslationTransform3D(),
    params0=None,
    static_grid2world=static_affine,
    moving_grid2world=moving_affine,
    starting_affine=c_of_mass.affine,
)

moving_translation = translation.transform(moving)
moving_translation_norm = normalize_for_display(moving_translation)

print("Translation affine:")
print(translation.affine)
print("\nRegistered shape:", moving_translation.shape)

## 12. Rigid coregistration — 6 degrees of freedom

Теперь разрешаем:

- 3 translations;
- 3 rotations.

Scaling и shear **не разрешаем**, потому что регистрируем изображения одного человека
и не хотим искусственно изменять размер / форму мозга.

In [ ]:
from dipy.align.transforms import RigidTransform3D

# Начальная точка для rigid optimization — уже оптимизированная translation.
rigid = affreg.optimize(
    static,
    moving,
    RigidTransform3D(),
    params0=None,
    static_grid2world=static_affine,
    moving_grid2world=moving_affine,
    starting_affine=translation.affine,
)

# Применяем итоговый rigid transform и resamples mean EPI в T1w grid.
moving_rigid = rigid.transform(moving)
moving_rigid_norm = normalize_for_display(moving_rigid)

print("Rigid affine:")
print(rigid.affine)
print("\nRegistered shape:", moving_rigid.shape)

В rigid affine:

- верхний левый `3×3` block кодирует rotation;
- последний столбец содержит translation component в world space;
- итоговый image всё равно остаётся resampled representation fMRI, а не «1 mm fMRI».

Небольшие off-diagonal значения в `3×3` block означают небольшой rotation.

## 13. Visual QC: translation vs rigid

Числовой optimization result нельзя принимать вслепую.  
Проверяем alignment визуально во всех трёх ортогональных плоскостях.

In [ ]:
def plot_registration_comparison(
    static_image,
    translation_image,
    rigid_image,
    alpha=0.35,
):
    """Сравнивает translation-only и rigid alignment в трёх плоскостях."""
    x, y, z = (np.array(static_image.shape) // 2).astype(int)

    slices = [
        ("Sagittal", (x, slice(None), slice(None))),
        ("Coronal", (slice(None), y, slice(None))),
        ("Axial", (slice(None), slice(None), z)),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    for col, (plane, slc) in enumerate(slices):
        # Верхний ряд: translation-only.
        axes[0, col].imshow(static_image[slc].T, cmap="gray", origin="lower")
        axes[0, col].imshow(
            translation_image[slc].T,
            cmap="magma",
            origin="lower",
            alpha=alpha,
        )
        axes[0, col].set_title(f"Translation — {plane}")

        # Нижний ряд: rigid.
        axes[1, col].imshow(static_image[slc].T, cmap="gray", origin="lower")
        axes[1, col].imshow(
            rigid_image[slc].T,
            cmap="magma",
            origin="lower",
            alpha=alpha,
        )
        axes[1, col].set_title(f"Rigid — {plane}")

    for ax in axes.ravel():
        ax.axis("off")

    plt.tight_layout()
    plt.show()


plot_registration_comparison(
    static_norm,
    moving_translation_norm,
    moving_rigid_norm,
)

В наших данных rigid alignment визуально улучшает совпадение, особенно в coronal и axial views.

Остаточные локальные различия около orbitofrontal cortex нельзя автоматически считать
ошибкой registration: EPI там особенно чувствительна к susceptibility distortion и signal dropout.

## 14. Final contour QC

Overlay может скрывать часть T1w anatomy.  
Поэтому дополнительно рисуем intensity contours mean EPI поверх T1w.

Эти contour levels **не являются границами конкретных тканей**.  
Это лишь уровни нормализованной EPI intensity для визуальной проверки геометрии.

In [ ]:
def plot_contour_qc(static_image, moving_image, levels=(0.25, 0.5, 0.75)):
    """Contour-based QC rigid registration в sagittal, coronal и axial views."""
    x, y, z = (np.array(static_image.shape) // 2).astype(int)

    slices = [
        ("Sagittal", (x, slice(None), slice(None))),
        ("Coronal", (slice(None), y, slice(None))),
        ("Axial", (slice(None), slice(None), z)),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    for ax, (plane, slc) in zip(axes, slices):
        ax.imshow(static_image[slc].T, cmap="gray", origin="lower")
        ax.contour(
            moving_image[slc].T,
            levels=levels,
            origin="lower",
        )
        ax.set_title(f"Rigid QC — {plane}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()


plot_contour_qc(static_norm, moving_rigid_norm)

## 15. What this notebook established

Практически пройдена цепочка:

`BIDS → NIfTI → voxel / volume / TR → affine → world coordinates → orientation → FOV → mean EPI → coregistration → resampling → QC`

Ключевые различия:

- **orientation** — направление image axes;
- **affine** — связь voxel coordinates с physical/world coordinates;
- **registration** — оценка transformation, совмещающего image content;
- **resampling** — применение transformation на target voxel grid;
- **coregistration** — здесь: functional EPI одного участника → его structural T1w;
- **normalization** — другой шаг: individual anatomy → common/template space, например MNI.

### Ограничения этого учебного pipeline

Мы **не** выполнили:

- motion correction / realignment 4D fMRI;
- susceptibility distortion correction;
- brain extraction / masking;
- slice-timing correction;
- spatial normalization;
- smoothing;
- statistical GLM.

Для будущего воспроизводимого fMRI analysis эти операции будут выполняться стандартным
preprocessing workflow (например, fMRIPrep), а этот notebook нужен, чтобы понимать,
что именно происходит под капотом.

### Next step

`events.tsv → temporal experimental structure → HRF → convolution → design matrix → GLM`